# ResNet-20 CIFAR-10 Fair QAT Training & Evaluation Notebook
### For Kaggle (T4 GPU) and Google Colab (T4 / A100 GPU)

This notebook reproduces and rigorously validates the FP8-BFP MLB-MAC architecture for ResNet-20 on CIFAR-10.

**Reviewer Concerns Resolved:**
1. **Reviewer 3 (Training Duration Disparity):** Matched-budget 50-epoch controls for FP32, Native FP8, and BFP MLB from scratch.
2. **Reviewer 2 (Exponent Sorting & Pairing):** Verification of offline static weight co-sorting and empirical ablation.

In [ ]:
# Cell 1: Environment Check
import torch
import os
import sys

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name : {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Please enable GPU in settings (Kaggle: GPU T4 x2, Colab: T4 GPU).")


In [ ]:
# Cell 2: Quick Smoke Test (Verify Pipeline in ~30 seconds)
!python train_experiments.py --mode full_suite --max-batches 3 --N 64


In [ ]:
# Cell 3: Matched-Budget Baseline Controls (Reviewer 3)
# 1. FP32 Baseline (50 epochs)
!python train_experiments.py --mode fp32_50 --epochs 50

# 2. Native FP8 Baseline (50 epochs - Matched Control)
!python train_experiments.py --mode fp8_50 --epochs 50

# 3. Native FP8 Baseline (30 epochs - Original Reproduction)
!python train_experiments.py --mode fp8_30 --epochs 30


In [ ]:
# Cell 4: Proposed BFP MLB Trained from Scratch across N
# 4a. 30-epoch Scratch Training (Paper Reproduction Control)
!python train_experiments.py --mode bfp_scratch_30 --epochs 30 --N 9,25,32,64,128,256,512

# 4b. 50-epoch Scratch Training (Matched Budget Control - No Fine-Tuning Confound)
!python train_experiments.py --mode bfp_scratch_50 --epochs 50 --N 9,25,32,64,128,256,512


In [ ]:
# Cell 5: Proposed BFP MLB Fine-Tuned (20 Epochs from 30-epoch FP8 Baseline Checkpoint)
# Note: Preserves fp8_30_seed{seed}_best.pth generated in Cell 3 without re-training or overwriting it.
!python train_experiments.py --mode bfp_finetune_20 --epochs 20 --N 9,25,32,64,128,256,512


In [ ]:
# Cell 6: Exponent Sorting Ablation at N=64 (Reviewer 2)
# Compares Sorted vs Unsorted BFP under identical 50-epoch scratch training
!python train_experiments.py --mode ablation_sorting --epochs 50 --N 64


In [ ]:
# Cell 7: Display Final Results Summary Table
import os
import pandas as pd
if os.path.exists('qat_summary_results.csv'):
    print('--- INDIVIDUAL RUN RESULTS ---')
    df = pd.read_csv('qat_summary_results.csv')
    display(df)
if os.path.exists('qat_aggregated_metrics.csv'):
    print('--- AGGREGATED STATISTICAL METRICS (MEAN +/- STD) ---')
    df_agg = pd.read_csv('qat_aggregated_metrics.csv')
    display(df_agg)
